# 01 — Generate the Logistics Q&A Dataset

Generates ~12,000 question/answer pairs using the Anthropic API.

**Before you run this notebook:**
1. Get an Anthropic API key from https://console.anthropic.com/
2. **Set a monthly spending cap** on the Anthropic console (Settings → Billing → Usage limits). Even $50 is enough; this run costs ~$5-15.
3. In Colab: Tools → User data → Add `ANTHROPIC_API_KEY`.

**Runtime:** ~30 minutes, costs ~$5-15 in Anthropic credits depending on model.

## Setup

In [ ]:
# Clone the repo and install deps
!git clone https://github.com/masonsau0/logistics-qa-lora.git
%cd logistics-qa-lora
!pip install -q anthropic datasets python-dotenv

In [ ]:
# Load API key from Colab secrets (Tools → User data)
import os

from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
assert os.environ["ANTHROPIC_API_KEY"].startswith("sk-ant-"), "Key looks malformed"

## Smoke test — verify the pipeline before spending money

In [ ]:
# Generates ~50 examples (1 batch per category) — should finish in ~1 minute and cost < $0.10
!python -m data.prepare_dataset --smoke --batch-size 8

In [ ]:
# Inspect a few generated examples
import json

with open("data/raw_generated.jsonl") as f:
    for line in list(f)[:3]:
        rec = json.loads(line)
        print(f"[{rec['category']}]")
        print(f"Q: {rec['question']}")
        print(f"A: {rec['answer'][:200]}...")
        print(f"Key facts: {rec['key_facts']}")
        print()

## Full run

Generates the full 12K-example dataset and writes the train/val/test splits.

Resumable — if Colab disconnects, just rerun. Already-generated examples are skipped.

In [ ]:
!python -m data.prepare_dataset --target 12000 --batch-size 8 --split

In [ ]:
# Verify split sizes
for split in ["train", "val", "test"]:
    n = sum(1 for _ in open(f"data/{split}.jsonl"))
    print(f"{split}: {n}")

## Save to Google Drive

Mount Drive and copy the splits so they survive between Colab sessions.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/logistics-qa-lora/data
!cp data/*.jsonl /content/drive/MyDrive/logistics-qa-lora/data/
!ls -lh /content/drive/MyDrive/logistics-qa-lora/data/

### Next step
Open `02_train_lora.ipynb`.